# 04. Retention & Cohort Analysis

In [ ]:
import pandas as pd
import os
import kagglehub
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

dataset_path = kagglehub.dataset_download('radistaleks/synthetic-bank-transactions')
categories    = pd.read_csv(os.path.join(dataset_path, 'categories.csv'))
clients       = pd.read_csv(os.path.join(dataset_path, 'clients.csv'))
subscriptions = pd.read_csv(os.path.join(dataset_path, 'subscriptions.csv'))
transactions  = pd.read_csv(os.path.join(dataset_path, 'transactions.csv'))

In [ ]:
clients['registration_date'] = pd.to_datetime(clients['registration_date'])
clients['birthdate']         = pd.to_datetime(clients['birthdate'])
subscriptions['date_start']  = pd.to_datetime(subscriptions['date_start'])
subscriptions['date_end']    = pd.to_datetime(subscriptions['date_end'])
transactions['date']         = pd.to_datetime(transactions['date'], format='%Y-%m-%d %H:%M:%S')

clients = clients.fillna(0)
subscriptions['product_company'] = subscriptions['product_company'].fillna('Неизвестно')
transactions['product_company']  = transactions['product_company'].fillna('Неизвестно')

transactions['month'] = transactions['date'].dt.to_period('M')

N = len(clients)

## 1. Когорты по году регистрации

In [ ]:
clients['reg_year'] = clients['registration_date'].dt.year
clients['reg_year'].value_counts().sort_index()

In [ ]:
# транзакции 2020 + год регистрации
txn = transactions.merge(clients[['id', 'reg_year']], left_on='client_id', right_on='id', how='left')

cohort = txn.groupby('reg_year').agg(
    users   = ('client_id', 'nunique'),
    count   = ('amount', 'count'),
    total   = ('amount', 'sum'),
    avg     = ('amount', 'mean'),
).sort_index()
cohort['txn_per_user'] = cohort['count'] / cohort['users']
cohort['arpu_year']    = cohort['total'] / cohort['users']
cohort

In [ ]:
cohort['arpu_year'].plot(kind='bar', figsize=(12, 4), title='ARPU за 2020 по когорте регистрации', rot=0)

In [ ]:
cohort['txn_per_user'].plot(kind='bar', figsize=(12, 4), title='Транзакций на клиента в 2020 по когорте', rot=0)

In [ ]:
cohort['avg'].plot(kind='bar', figsize=(12, 4), title='Средний чек в 2020 по когорте', rot=0)

## 2. Retention Matrix по подпискам (когорта = год старта)

In [ ]:
subs = subscriptions.copy()
OBS_END = pd.Timestamp('2020-12-31')

subs['end_filled']    = subs['date_end'].fillna(OBS_END)
subs['duration_days'] = (subs['end_filled'] - subs['date_start']).dt.days
subs['cohort_year']   = subs['date_start'].dt.year

milestones = [0, 90, 180, 365, 730, 1095, 1460, 2190]
m_labels   = ['0д', '3м', '6м', '1г', '2г', '3г', '4г', '6г']

# для каждой когорты считаем % выживших
rows = {}
for year in sorted(subs['cohort_year'].unique()):
    group = subs[subs['cohort_year'] == year]
    rows[year] = {lbl: round((group['duration_days'] >= d).mean() * 100, 1)
                  for d, lbl in zip(milestones, m_labels)}

retention = pd.DataFrame(rows).T
retention

In [ ]:
plt.figure(figsize=(14, 7))
sns.heatmap(retention, annot=True, fmt='.0f', cmap='Blues',
            linewidths=0.5, vmin=0, vmax=100,
            cbar_kws={'label': '% активных подписок'})
plt.title('Retention Matrix: когорта (год старта) × длительность')
plt.tight_layout()

## 3. Анализ оттока подписок

In [ ]:
churned = subscriptions[subscriptions['date_end'].notna()].copy()
churned['duration_days'] = (churned['date_end'] - churned['date_start']).dt.days
churned['churn_month']   = churned['date_end'].dt.to_period('M')

print('Отменённых подписок:', len(churned))
churned['duration_days'].describe()

In [ ]:
churned.groupby('churn_month').size().sort_index().plot(
    kind='bar', figsize=(12, 4), title='Отменённые подписки по месяцам', rot=45
)

In [ ]:
churned['duration_days'].hist(bins=30, figsize=(10, 4), edgecolor='white')
plt.title('Распределение длительности до отмены, дней')

In [ ]:
# churn rate по категории подписки
subscriptions.groupby('product_category').apply(
    lambda g: pd.Series({'total': len(g), 'churned': g['date_end'].notna().sum(),
                         'churn_rate': round(g['date_end'].notna().mean() * 100, 1)})
)

In [ ]:
# медианная длительность до отмены по сервисам
churned[churned['product_category'] == 4].groupby('product_company')['duration_days'].agg(
    ['count', 'mean', 'median']
).sort_values('median', ascending=False).round(0)

In [ ]:
churned[churned['product_category'] == 4].groupby('product_company')['duration_days'].median(
).sort_values().plot(kind='barh', figsize=(10, 5), title='Медиана дней до отмены по музыкальному сервису')

## 4. LTV подписок

In [ ]:
subs['duration_months'] = (subs['duration_days'] / 30).clip(lower=1)
subs['ltv']       = subs['amount'] * subs['duration_months']
subs['is_active'] = subs['date_end'].isna()

subs.groupby('product_category')[['ltv', 'duration_months']].agg(['mean', 'median']).round(0)

In [ ]:
music_ltv = (
    subs[subs['product_category'] == 4]
    .groupby('product_company')
    .agg(count=('ltv','count'), avg_ltv=('ltv','mean'), active_pct=('is_active','mean'))
    .sort_values('avg_ltv', ascending=False)
)
music_ltv['active_pct'] = (music_ltv['active_pct'] * 100).round(1)
music_ltv.round(0)

In [ ]:
music_ltv['avg_ltv'].sort_values().plot(kind='barh', figsize=(10, 5), title='Средний LTV по музыкальному сервису')

## 5. Retention curve по транзакциям

In [ ]:
# первый месяц каждого клиента в 2020
first_month = transactions.groupby('client_id')['month'].min().rename('first_month')
txn2 = transactions.merge(first_month, on='client_id', how='left')
txn2['offset'] = (
    txn2['month'].dt.to_timestamp() - txn2['first_month'].dt.to_timestamp()
).dt.days // 30

curve = txn2[txn2['offset'] <= 11].groupby('offset')['client_id'].nunique()
(curve / N * 100).round(1)

In [ ]:
(curve / N * 100).plot(marker='o', figsize=(12, 5), title='Retention по транзакциям 2020')
plt.xlabel('Месяц от первой транзакции')
plt.ylabel('% активных клиентов')